# <div align="center"> Statistical Learning </div>
# <div align="center"> Machine Learning and Econometrics </div>
## <div align="center"> ECO 4971/6971 </div>
#### <div align="center">Class 13 — Unsupervised Learning</div>
<div align="center"> Jonathan Holmes (he/him)</div>

# Road Map

$$Y = f(X) + \varepsilon$$

Every method we have studied so far assumes we observe the outcome $Y$. Today we drop that assumption entirely. In **unsupervised learning** there is no response variable — only a matrix of features $\mathbf{X}$. Our goal shifts from *prediction* to *discovery*: finding hidden structure in the data.

By the end of this class you will be able to:

- Explain what unsupervised learning is and why it is useful.
- Apply **K-means clustering** to partition observations into $K$ groups by minimizing within-cluster variation.
- Build and interpret a **hierarchical clustering dendrogram** without pre-specifying the number of clusters.
- Use **principal component analysis (PCA)** to reduce the dimensionality of a dataset while retaining as much variance as possible.
- Understand why and when we need to scale variables before applying these methods.

# Unsupervised Learning

Every method we have studied so far has been an example of **supervised learning**: we observed labelled pairs $\{Y_i, X_i\}_{i=1}^n$ and trained a model $\hat{f}$ to predict $Y$ from $X$. The label $Y$ *supervised* the learning — it told the algorithm what a good prediction looked like.

In **unsupervised learning** there is no response variable. We observe only the features $X_1, X_2, \dots, X_p$ for $n$ observations, and the goal is to discover interesting structure in the data: natural groupings, low-dimensional representations, or unusual patterns. Without a $Y$ to evaluate predictions against, there is no standard measure of success — which makes these methods harder to assess and more exploratory in nature.

## Why Use Unsupervised Learning?

If we have no outcome to predict, why bother? Often, the structure *in the features themselves* is exactly what we care about. Consider an e-commerce retailer trying to personalise product recommendations. The retailer has no labelled training data of the form "this customer would enjoy that product" — only a purchase and browsing history for each user. By grouping customers with similar histories together, the algorithm can recommend products that similar customers have bought, without ever observing a direct outcome. This is the logic behind collaborative filtering, the engine behind most recommendation systems.

More generally, unsupervised methods are useful whenever we want to understand the natural groupings in a dataset, compress high-dimensional data for visualisation, or detect anomalies — all without requiring labelled examples.

# Part 1: Clustering Methods

**Clustering** refers to a broad family of techniques for finding subgroups, or **clusters**, among the observations in a dataset. We seek to partition observations into distinct groups such that observations within each group are similar to one another, while observations in different groups are quite different. What "similar" means is a domain-specific choice — usually Euclidean distance in feature space, but other metrics are possible.

We will study two classical algorithms: **K-means clustering**, which requires us to specify the number of clusters $K$ in advance, and **hierarchical clustering**, which builds a tree of nested cluster merges that can be cut at any level to yield any desired number of groups.

## K-Means Clustering

**K-means clustering** partitions the dataset into $K$ distinct, non-overlapping clusters. The algorithm requires only one input from the analyst: the number of clusters $K$. Given $K$, each of the $n$ observations is assigned to exactly one of the $K$ clusters.

Before working through the math, let's see it in action on a simulated dataset where we know the true group labels — this lets us judge how well the algorithm recovers structure we built in by hand.

In [ ]:
# data
import pandas as pd
import numpy as np

# visualization
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="white")

# ML
from sklearn.datasets import make_blobs
from sklearn.cluster import KMeans

# create simulated dataset: 150 observations, 2 features, 4 true groups
X, y = make_blobs(n_samples=150, n_features=2, centers=4, cluster_std=2, random_state=17)
df = pd.DataFrame(columns=['x1', 'x2'], data=X)
df['group'] = y
df.sort_values('group', inplace=True)

df.head()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: unlabeled — what the algorithm actually sees
sns.scatterplot(x='x1', y='x2', color='k', s=80, data=df, ax=axes[0])
axes[0].set_title('Unlabeled data — what the algorithm sees', fontsize=12)

# Right: true labels for reference
sns.scatterplot(x='x1', y='x2', hue='group', s=80, palette='tab10', data=df, ax=axes[1])
axes[1].set_title('True group labels (hidden from the algorithm)', fontsize=12)

plt.tight_layout()
plt.show()

In [ ]:
x = df[['x1', 'x2']]

# Fit K-means with K=2 — the algorithm doesn't know the true number of groups is 4
km2 = KMeans(n_clusters=2, n_init=20, random_state=1706)
km2.fit(x)
df['Predicted Group'] = km2.labels_

fig, ax = plt.subplots(1, 1, figsize=(8, 8))
sns.scatterplot(x='x1', y='x2', s=80, hue='Predicted Group', palette='tab10', data=df, ax=ax)
ax.set_title('K-means with K=2', fontsize=12)
plt.show()

In [ ]:
k_list = [2, 3, 4]
x = df[['x1', 'x2']]
fig, axes = plt.subplots(1, len(k_list), figsize=(20, 6))

for i, k in enumerate(k_list):
    km = KMeans(n_clusters=k, n_init=20, random_state=1706)
    km.fit(x)
    pred = f'Predicted Group - k={k}'
    df[pred] = km.labels_
    sns.scatterplot(x='x1', y='x2', hue=pred, data=df, ax=axes[i], palette='tab10')
    axes[i].set_title(f'K = {k}', fontsize=12)

plt.tight_layout()
plt.show()

In [ ]:
# Confusion matrix: actual group (rows) vs K=4 predicted group (columns)
print("Before label alignment:")
display(pd.crosstab(df['group'], df['Predicted Group - k=4'],
                    rownames=['Actual Group'], colnames=['Predicted Group (K=4)']))

# K-means labels are arbitrary integers — swap 0↔2 to align with the true group labels
df['Predicted Group - k=4'].replace({2: 0, 0: 2}, inplace=True)

print("After label alignment:")
display(pd.crosstab(df['group'], df['Predicted Group - k=4'],
                    rownames=['Actual Group'], colnames=['Predicted Group (K=4)']))

In [ ]:
df.loc[df.group==df['Predicted Group - k=4'],'Classification']= 'Correct'
df.loc[df.group!=df['Predicted Group - k=4'],'Classification']= 'False'

fig, axes =  plt.subplots(1,2, figsize=(20,10))

sns.scatterplot(x='x1',y='x2', hue='Predicted Group - k=4', style='group',data=df, ax=axes[0] )
sns.scatterplot(x='x1',y='x2', color='k',alpha=.5,data=df, ax=axes[1] )
sns.scatterplot(x='x1',y='x2', hue='Classification', style='group',data=df, ax=axes[1] )

plt.show()

## k-means explained:  notation
- Let $C_1, \dots, C_K$  denote sets containing the indices of the observations in each cluster. 
- These sets satisfy two properties:
    1. $C_1\cup C_2 \cup \dots \cup C_K = \{1, \dots, n\}$. In other words, each observation belongs to at least one of the K clusters.
    2. $C_k ∩ C_{k'} = \emptyset$ for all $k \neq k'$. In other words, the clusters are nonoverlapping: no observation belongs to more than one cluster.
    - if the $i_{th}$ observation is in the $k_{th}$ cluster, then $i \in C_k$.
- A good clustering is one for which the within-cluster variation is as small as possible
- Let the measure of within-cluster variation for cluster k be given by $W(C_k)$

## K-Means: The Minimization Problem

$$\min_{C_1, \dots, C_K} \left\{ \sum_{k=1}^K W(C_k) \right\}$$

We want to partition the observations into $K$ clusters such that the total **within-cluster variation**, summed over all $K$ clusters, is as small as possible. The most common choice of $W(C_k)$ is the **squared Euclidean distance** — conceptually similar to minimizing mean squared error, but without a response variable $Y$. We are measuring how spread out each cluster is around its own centre.

## k-means explained: Euclidean distance
$$ \large W(C_k) = \frac{1}{\left|C_k\right|} \sum_{i, i' \in C_k} \sum_{j=1}^p(x_{ij} - x_{i'j})^2$$
- where $\left|C_k\right|$ denotes the number of observations in the $k_{th}$ cluster. 
- This is the sum of all of the pairwise squared Euclidean distances between the observations in the $k_{th}$ cluster, divided by the total number of observations in the $k_{th}$ cluster.


In [ ]:
import matplotlib.image as mpimg
img = mpimg.imread('euclidian_distance.png')
plt.imshow(img)

## k-means explained: solution
Combining the last two equations yields the optimization problem that defines K-means clustering:

$$\min_{C_1, \dots, C_K} \left\{ \sum_{k=1}^K \frac{1}{\left|C_k\right|} 
\sum_{i, i' \in C_k} \sum_{j=1}^p(x_{ij} - x_{i'j})^2 \right\}$$

## K-Means: The Algorithm

The K-means algorithm is simple and fast:

1. **Random initialization.** Randomly assign each of the $n$ observations a label from $1$ to $K$. These are the initial cluster assignments.
2. **Iterate until convergence.** Repeat the following two steps until the cluster assignments stop changing:
   - (a) *Centroid step.* For each of the $K$ clusters, compute the cluster **centroid** — the vector of feature means for all observations currently in that cluster.
   - (b) *Assignment step.* Reassign each observation to the cluster whose centroid is closest in Euclidean distance.

Each step is guaranteed to decrease the objective, so the algorithm always converges. However, it may converge to a **local minimum** rather than the global minimum. In practice we run the algorithm multiple times from different random starts (`n_init` in scikit-learn) and keep the solution with the lowest total within-cluster variation.

## Why the Algorithm Works

The key insight is that within-cluster variation around the centroid is equivalent (up to a factor of 2) to the sum of pairwise distances within the cluster:

$$ \frac{1}{\left|C_k\right|} \sum_{i, i' \in C_k} \sum_{j=1}^p(x_{ij} - x_{i'j})^2 = 
2 \sum_{i \in C_k} \sum_{j=1}^p(x_{ij} - \bar{x}_{kj})^2$$

where $\bar{x}_{kj} = \frac{1}{|C_k|} \sum_{i \in C_k} x_{ij}$ is the mean of feature $j$ in cluster $C_k$. This identity shows why the algorithm makes progress at each iteration: the centroid step minimizes the right-hand side over all possible centroids, and the assignment step minimizes it over all cluster assignments. Together, each full iteration can only decrease the total within-cluster variation.

![](sphx_glr_plot_kmeans_digits_001.png)
Source: https://scikit-learn.org/stable/auto_examples/cluster/plot_kmeans_digits.html

## In-Class Exercise #1

**K-Means Clustering**

Q1: In the K = 2 solution above, the algorithm splits the data into two groups even though the true data has four clusters. Why does K-means produce a result like this when $K$ is misspecified? What would you expect to happen to total within-cluster variation as $K$ increases toward 4?

Q2: The K-means algorithm starts from a random initialisation. Why might two runs with the same $K$ return different cluster assignments? How does the `n_init` parameter address this?

Q3: Modify the code to try $K = 5$ and $K = 6$. Does the solution look sensible? What does this suggest about the challenge of choosing $K$?

In [ ]:
# Your code here


# Part 2: Hierarchical Clustering

A key limitation of K-means clustering is that we must commit to a number of clusters $K$ before seeing the data. **Hierarchical clustering** avoids this by building a complete tree of nested cluster merges, called a **dendrogram**, which can then be "cut" at any height to produce any desired number of clusters.

The most common variant is **agglomerative** (bottom-up) clustering. We start by treating every observation as its own cluster (the leaves of the tree), then repeatedly merge the two most similar clusters until only one cluster remains (the trunk). The result is a dendrogram that encodes the full merge history — like the decision tree structures we saw in Class 11, except here the tree is built bottom-up from the data.

### Linkage: measuring distance between groups

When merging clusters, we need a way to measure the distance between two *groups* of observations, not just two individual points. The choice of **linkage method** affects the shape of the dendrogram:

| Linkage | Distance between clusters $A$ and $B$ |
| --- | --- |
| **Complete** | Maximum pairwise distance: $\max_{i \in A,\, i' \in B} d(i, i')$ |
| **Single** | Minimum pairwise distance: $\min_{i \in A,\, i' \in B} d(i, i')$ |
| **Average** | Mean pairwise distance: $\frac{1}{\lvert A \rvert \cdot \lvert B \rvert}\sum_{i \in A}\sum_{i' \in B} d(i, i')$ |

Complete and average linkage tend to produce more balanced, compact dendrograms and are the most widely used in practice. We use complete linkage below.

In [ ]:
# create simulated dataset based on two features and 4 distinct groups
X, y = make_blobs(n_samples=45, n_features=2, centers=3, cluster_std=2 , random_state=1)
df=pd.DataFrame(columns=['x1','x2'], data=X)
df['group']=y
display(df.head())

txt="Forty-five observations generated in two-dimensional space.\nIn reality there are three distinct classes, shown in separate colors.\nHowever, we will treat these class labels as unknown and will seek to cluster\nthe observations in order to discover the classes from the data."
fig, ax =  plt.subplots(1,1, figsize=(8,8))
sns.scatterplot(x='x1',y='x2',hue='group',s=80,legend=False,data=df)
fig.text(0.5, -.05, txt, ha='center', wrap=True, fontsize=12)

plt.show()

In [ ]:
from scipy.cluster.hierarchy import linkage, dendrogram
mergings = linkage(df[['x1','x2']], method='complete')

fig, ax=plt.subplots(1,1,figsize=(10,10))
dendrogram(mergings,leaf_rotation=90,leaf_font_size=10, ax=ax)

plt.show()


In [ ]:

    
fig, ax =  plt.subplots(1,1, figsize=(8,8))
sns.scatterplot(x='x1',y='x2',hue='group',alpha=0.5,legend=False,data=df)
for i in df.index:
    ax.annotate(i, (df.x1.loc[i], df.x2.loc[i]), ha='center')
plt.show()

## Interpreting a Dendrogram

Each **leaf** at the bottom of the dendrogram represents a single observation. As we move up the tree, leaves begin to fuse into **branches** — each fusion corresponds to a merge of two clusters. Two key rules:

- **Height of fusion = dissimilarity.** The higher up in the tree that two observations first meet, the more dissimilar they are. Observations that fuse early (near the bottom) are very similar; those that fuse late (near the top) are quite different.
- **Cutting the tree gives clusters.** Drawing a horizontal line at height $h$ produces as many clusters as there are vertical lines crossing that cut. Choosing the cut height plays the same role as choosing $K$ in K-means.

A useful heuristic for choosing the cut: look for the largest vertical gap in the dendrogram that is *not* already crossed by a horizontal line — that gap suggests where a natural number of clusters lies. Cross-referencing with the scatter plot above (where we labeled the observations by their index) lets you verify whether the clusters make geometric sense.

## In-Class Exercise #2

**Hierarchical Clustering**

Q1: In the dendrogram above, identify two observations that are very similar (fuse near the bottom of the tree) and two that are very dissimilar (fuse near the top). Cross-reference with the labelled scatter plot — do these pairs look close or far apart in feature space?

Q2: At approximately what height would you cut the dendrogram to recover three clusters? How does this partition compare to K-means with $K = 3$?

Q3: We used *complete linkage* above. Change `method='complete'` to `method='single'` and replot the dendrogram. How does the shape change, and why? (Hint: think about what single linkage does when one observation is close to a large cluster.)

In [ ]:
# Your code here


# Part 3: Principal Component Analysis

Clustering finds homogeneous subgroups among *observations*. **Principal component analysis (PCA)** tackles a complementary problem: finding a low-dimensional representation of the *features* that captures as much variation as possible.

PCA is an unsupervised method — it uses only $X_1, X_2, \dots, X_p$ with no response $Y$. Given an $n \times p$ data matrix $\mathbf{X}$, PCA constructs a sequence of new variables — the **principal components** — that are linear combinations of the original features, ordered so that the first component explains the most variance, the second explains the most remaining variance, and so on.

The main uses of PCA are:

- **Visualization**: project high-dimensional data onto two or three components to create a readable plot.
- **Dimensionality reduction**: replace $p$ correlated features with a smaller number of uncorrelated components before fitting a supervised model.
- **Exploratory analysis**: identify which features drive the most variation in the data.

## What Are Principal Components?

Suppose we want to visualise $n$ observations measured on $p$ features $X_1, X_2, \dots, X_p$. One approach is to examine all pairwise scatterplots. But there are $\binom{p}{2} = p(p-1)/2$ such plots — with $p = 10$ that is 45 plots, and with $p = 100$ it is nearly 5,000. Even if we could look at all of them, each plot captures only a small fraction of the total information in the data.

PCA solves this by finding the directions in $p$-dimensional feature space along which the data vary the most. Projecting the observations onto these directions — the principal components — gives a low-dimensional summary that retains as much information as possible. With $p = 4$ features the pair plot below is still manageable, but the biplot we produce at the end of this section conveys the same structure far more efficiently.

In [ ]:
df=pd.read_csv("USArrests.csv", index_col=0)
display(df.head())
df.info()

In [ ]:
sns.pairplot(df)

## First principal component
- The first principal component of a set of features $X_1,X_2, . . . , X_p$ is the normalized linear combination of the features, that has the largest variance.
$$Z_1 = \phi_{11}X_1 + \phi_{21}X_2 + \dots + \phi_{p1}X_p$$
- Normalized means: $\sum_{j=1}^p\phi_{j1}^2=1$, their sum of squares is equal to one
    - otherwise setting these elements to be arbitrarily large in absolute value could result in an arbitrarily large variance
- $\phi_{11}, \dots, \phi_{p1}$ are called __loadings__ of the principal first component
    - $(\phi_{11}, \dots, \phi_{p1})^T = $ the principal component loading vector $\phi_{1}$

## Computing the First Principal Component

Since we care only about variance, we first centre each variable to have mean zero. We then look for the linear combination of the sample feature values of the form:

$$z_{i1} = \phi_{11}x_{i1} + \phi_{21}x_{i2} + \dots + \phi_{p1}x_{ip}$$

The first principal component loading vector $\boldsymbol{\phi}_1 = (\phi_{11}, \dots, \phi_{p1})^\top$ solves:

$$\max_{\phi_{11}, \dots, \phi_{p1}} \left\{\frac{1}{n} \sum_{i=1}^n \left( \sum_{j=1}^p \phi_{j1}x_{ij} \right)^2 \right\} \quad \text{s.t.} \quad \sum_{j=1}^p\phi_{j1}^2=1$$

In words: find the unit-length direction $\boldsymbol{\phi}_1$ that maximises the **variance** of the projected scores $z_{i1}$. The constraint $\sum_j \phi_{j1}^2 = 1$ prevents a trivial solution where the loadings are made arbitrarily large.

## Geometric Interpretation

The loading vector $\boldsymbol{\phi}_1$ defines a **direction in feature space** along which the data vary the most. Each observation's **score** $z_{i1}$ is its coordinate when projected onto this direction.

The figure below illustrates this for a two-variable example. The first principal component is the line through the cloud of points that minimises the total squared perpendicular distance from each point to the line — or equivalently, that maximises the variance of the projections along the line.

![](figure6_4.png)
*The first principal component direction (solid line) is the axis of maximum variance. Each observation's score $z_{i1}$ is its signed distance along this axis from the origin.*

## The Second Principal Component

The **second principal component** $Z_2$ is the linear combination of $X_1, \dots, X_p$ with the largest variance among all combinations that are **uncorrelated with $Z_1$**:

$$z_{i2} = \phi_{12}x_{i1} + \phi_{22}x_{i2} + \dots + \phi_{p2}x_{ip}$$

The uncorrelated constraint is equivalent to requiring $\boldsymbol{\phi}_2 \perp \boldsymbol{\phi}_1$ — the second loading vector must be orthogonal (perpendicular) to the first. Successive components $Z_3, Z_4, \dots$ are defined in the same way: each captures the maximum remaining variance subject to orthogonality with all previous components. With $p$ features there are at most $\min(n-1, p)$ principal components in total.

Once we have computed the components, we can plot observations against any two of them to get a low-dimensional view of the data.

## Illustration: USArrests

To illustrate PCA in practice, we use the `USArrests` dataset — one of our recurring ISLR datasets. It contains four variables for each of the 50 U.S. states:

- **Murder**, **Assault**, and **Rape**: arrests per 100,000 residents for each crime.
- **UrbanPop**: the percentage of the state population living in urban areas.

With $n = 50$ observations and $p = 4$ features, there are $\binom{4}{2} = 6$ possible scatterplots but only four principal components. We will reduce this to two principal components and visualise all 50 states in a single **biplot** — a plot that shows both the observation scores and the feature loading vectors simultaneously.

In [ ]:
display(df.head())
df.describe().T.loc[:,['mean','std']]

In [ ]:
from sklearn.preprocessing import scale

X = pd.DataFrame(scale(df), index=df.index, columns=df.columns)
X.describe().T.loc[:,['mean','std']] # mean zero and std of 1

In [ ]:
from sklearn.decomposition import PCA
# The loading vectors
pca_loadings = pd.DataFrame(PCA().fit(X).components_.T, index=df.columns, columns=[r'$\phi_1$', r'$\phi_2$', r'$\phi_3$', r'$\phi_4$'])
pca_loadings

In [ ]:
np.square(pca_loadings).sum().reset_index().rename(columns={'index': 'Loading Vector', 0:"Loadings sum of squares"})

In [ ]:
# Fit the PCA model and transform X to get the principal components
pca = PCA()
df_plot = pd.DataFrame(pca.fit_transform(X), columns=['PC1', 'PC2', 'PC3', 'PC4'], index=X.index)
df_plot

## Proportion of Variance Explained

Once we have computed the principal components, a natural question is: how many do we actually need? The **proportion of variance explained** (PVE) by each component answers this. The **scree plot** displays PVE on the vertical axis and the component number on the horizontal axis — we look for an "elbow" where adding another component yields diminishing returns.

In [ ]:
pve = pd.DataFrame({
    'PC': [f'PC{i+1}' for i in range(4)],
    'PVE': pca.explained_variance_ratio_,
    'Cumulative PVE': pca.explained_variance_ratio_.cumsum()
})
display(pve)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].bar(pve['PC'], pve['PVE'])
axes[0].set_ylabel('Proportion of Variance Explained', fontsize=12)
axes[0].set_title('Scree Plot', fontsize=12)

axes[1].plot(pve['PC'], pve['Cumulative PVE'], marker='o')
axes[1].set_ylabel('Cumulative PVE', fontsize=12)
axes[1].set_title('Cumulative Variance Explained', fontsize=12)
axes[1].set_ylim(0, 1.05)

plt.tight_layout()
plt.show()

In [ ]:
fig, ax1 = plt.subplots(figsize=(12, 12))

ax1.set_xlim(-3.5, 3.5)
ax1.set_ylim(-3.5, 3.5)

# Plot PC1 and PC2 scores for each state
for i in df_plot.index:
    ax1.annotate(i, (df_plot.PC1.loc[i], -df_plot.PC2.loc[i]), ha='center', color='darkgreen')

# Reference lines at the origin
ax1.hlines(0, -3.5, 3.5, linestyles='dotted', colors='grey')
ax1.vlines(0, -3.5, 3.5, linestyles='dotted', colors='grey')

ax1.set_xlabel('First Principal Component', fontsize=16)
ax1.set_ylabel('Second Principal Component', fontsize=16)

# Overlay loading vectors on a second axis (scaled to [-1, 1])
ax2 = ax1.twinx().twiny()
ax2.set_ylim(-1, 1)
ax2.set_xlim(-1, 1)
ax2.tick_params(axis='y', colors='orange')
ax2.set_xlabel('Principal Component loading vectors', color='darkorange')

# Small offset so arrow tip and label don't overlap
a = 1.07
for i in pca_loadings[[r'$\phi_1$', r'$\phi_2$']].index:
    ax2.annotate(i, (pca_loadings[r'$\phi_1$'].loc[i] * a, -pca_loadings[r'$\phi_2$'].loc[i] * a), color='darkorange')

# Draw all four loading vectors
ax2.arrow(0, 0, pca_loadings[r'$\phi_1$'][0], -pca_loadings[r'$\phi_2$'][0], color='darkorange')
ax2.arrow(0, 0, pca_loadings[r'$\phi_1$'][1], -pca_loadings[r'$\phi_2$'][1], color='darkorange')
ax2.arrow(0, 0, pca_loadings[r'$\phi_1$'][2], -pca_loadings[r'$\phi_2$'][2], color='darkorange')
ax2.arrow(0, 0, pca_loadings[r'$\phi_1$'][3], -pca_loadings[r'$\phi_2$'][3], color='darkorange')

txt = ("Biplot for the USArrests data. Green labels show the first two principal component scores for each state.\n"
       "Orange arrows show the loading vectors for all four variables (axes on top and right).\n"
       "The direction and length of each arrow indicate how strongly that variable loads onto each component.")
fig.text(0.5, -0.03, txt, ha='center', wrap=True, fontsize=12)

plt.show()

## Why Did We Scale the Variables?

The figure above shows what happens when we apply PCA to the *unscaled* `USArrests` data. Because `Assault` has a much larger variance than the other variables (assault arrests are far more common than murders or rapes), it completely dominates the first principal component — the other three variables contribute almost nothing. After scaling each variable to have mean zero and standard deviation one, the principal components reflect genuine structure in the data rather than arbitrary differences in measurement units or scale.

**Rule of thumb:** always scale your variables before applying PCA unless they are already measured in the same units and you have a specific reason to let high-variance features dominate.

**Try it:** Re-run the PCA cells above after commenting out the `scale()` call. Compare the loading vectors — notice how `Assault` swamps the first component without scaling.

## In-Class Exercise #3

**Principal Component Analysis**

Q1: Look at the `pca_loadings` table. Murder, Assault, and Rape all load heavily onto the first principal component, while UrbanPop loads more onto the second. What does each principal component appear to be measuring in terms of the original variables?

Q2: In the biplot, states like Florida, Nevada, and California appear far to the right (high PC1 scores). What does a high PC1 score imply about a state's crime profile?

Q3: The scree plot shows that PC1 alone explains a large fraction of the variance. If you were building a regression model to predict something about U.S. states, at what point would you stop adding principal components? What criterion would you use?

In [ ]:
# Your code here
